# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

It is wrong because the API sent JSON as a string of text, and Python only turned it into a dictionary after using json.loads().


### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

# TODO: the portfolio name
print("portfolio name:", data["portfolio"]["name"])
# TODO: how many holdings?
print("number of holdings:", len(data["portfolio"]["holdings"]))
# TODO: how many price records in total? (len() on each holding's prices)
print("total price records:", sum(len(h["prices"]) for h in data["portfolio"]["holdings"]))

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio name: Cavalier Growth
number of holdings: 3
total price records: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: _3_
- price rows: _9_
- one row of the table you are about to build is one _price record_

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
print(data['portfolio']['holdings'][0]['company']['sector'])

# TODO: now try the same thing for KO -- what happens, and why?
company = data['portfolio']['holdings'][2]['company']   # KeyError because KO's company dictionary does not have a 'sector' key.
# TODO: rewrite it using an `in` check so a missing sector prints 'unknown'
if 'sector' in company:
    print(company['sector'])
else:
    print('unknown')

Technology
unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [5]:
# TODO: loop over holdings, then over prices, and report the four problems
#
# Hint for #2: isinstance(value, str)
# Hint for #3: write it as `volume == 0`, not `if not volume` -- ask yourself why

for holding in data['portfolio']['holdings']:
    ticker = holding['ticker']

    for price in holding['prices']:
        date = price['date']
        if price['close'] is None:
          print('None close:', ticker, date)
        if isinstance(price['close'], str):
          print('String close:', ticker, date)
        if price['volume'] == 0:
          print('Zero volume:', ticker, date)
    if 'sector' not in holding['company']:
      print('Missing sector:', ticker)

String close: MSFT 2026-09-29
Zero volume: MSFT 2026-09-29
None close: KO 2026-09-29
Missing sector: KO


**TODO:** your answer to the last question: _The KO None close and missing sector is the missing data. The string close and zero volume MSFT are real measurements. I think the string close isn't the right data type._

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
# TODO: px = pd.json_normalize(holdings, record_path=..., meta=[...], sep='.',
#                              errors='ignore')
holdings = data['portfolio']['holdings']
px = pd.json_normalize(
    holdings,
    record_path='prices',
    meta=['ticker', 'shares'],
    errors='ignore'
)
assert len(px) == 9
# TODO: print px.dtypes and find the column that is the wrong type
print(px.dtypes)

date      object
close     object
volume     int64
ticker    object
shares    object
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
# TODO: px['close'] = pd.to_numeric(...)
# TODO: check the dtype and the null count

px['close'] = pd.to_numeric(px['close'], errors='coerce')
print(px['close'].dtypes)
print('Null count:', px['close'].isna().sum()) # null came from KO's missing close value (None) and not from the MSFT string

float64
Null count: 1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
px['close'] = pd.to_numeric(px['close'], errors='coerce')
# TODO 1 -- mean close and day count per ticker
mean_close = px.groupby('ticker')['close'].agg(['mean', 'count'])
print(mean_close)


# TODO 2 -- position values on 2026-09-30, total, and largest
last_day = px[px['date'] == '2026-09-30'].copy()

shares = {
    h['ticker']: h['shares']
    for h in data['portfolio']['holdings']
}

last_day['shares'] = last_day['ticker'].map(shares)
last_day['position_value'] = last_day['shares'] * last_day['close']

print(last_day[['ticker', 'shares', 'close', 'position_value']])

print('Total portfolio value:', last_day['position_value'].sum())

largest = last_day.loc[last_day['position_value'].idxmax()]
print('Largest holding:', largest['ticker'])


# TODO 3 -- mean volume with and without the zero day
with_zero = px['volume'].mean()
without_zero = px.loc[px['volume'] != 0, 'volume'].mean()

print('Mean volume with zero:', with_zero)
print('Count with zero:', px['volume'].count())

print('Mean volume without zero:', without_zero)
print('Count without zero:', px.loc[px['volume'] != 0, 'volume'].count())

print('Difference:', without_zero - with_zero)



           mean  count
ticker                
AAPL    231.450      3
KO       71.075      2
MSFT    502.050      3
  ticker  shares  close  position_value
2   AAPL     120  233.1         27972.0
5   MSFT      45  505.3         22738.5
8     KO     300   70.9         21270.0
Total portfolio value: 71980.5
Largest holding: AAPL
Mean volume with zero: 26677777.777777776
Count with zero: 9
Mean volume without zero: 30012500.0
Count without zero: 8
Difference: 3334722.222222224


**TODO:** the zero-volume question: _Dropping the zero was not the right call because zero is a real recorded measurement, not missing data, and removing it would increase the average volume_

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
# TODO: build the result dict, json.dump it, and print the file back
# Build the result dictionary
result = {
    'date': data['as_of'],
    'currency': data['currency'],
    'summary': mean_close.reset_index().to_dict(orient='records')
}

with open('portfolio_summary.json', 'w') as f:
    json.dump(result, f, indent=2, allow_nan=False)
with open('portfolio_summary.json', 'r') as f:
    print(f.read())

{
  "date": "2026-09-30",
  "currency": "USD",
  "summary": [
    {
      "ticker": "AAPL",
      "mean": 231.45000000000002,
      "count": 3
    },
    {
      "ticker": "KO",
      "mean": 71.075,
      "count": 2
    },
    {
      "ticker": "MSFT",
      "mean": 502.05,
      "count": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [10]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [11]:
# TODO A, B, C, D

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
holdings_count = 3      # TODO: should be 3
price_rows = 9         # TODO: should be 9
largest_position = 'AAPL'   # TODO: which ticker, on 2026-09-30
null_vs_zero = 'None close was missing data and the zero volume was a real measurement'       # TODO: which value was missing, which was a real measurement
zero_volume_call = 'I kept the zero in the mean because it was a real measurement. If I took it out, it would incorrectly inflate the average'   # TODO: did you keep the zero in the mean, and why

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: None close was missing data and the zero volume was a real measurement
zero volume: I kept the zero in the mean because it was a real measurement. If I took it out, it would incorrectly inflate the average
